In [ ]:
import torch
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
import numpy as np
import random
import os
import pandas as pd
import torch.nn as nn
from collections import defaultdict
import seaborn as sns

# from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.integrate import simps
import pickle
import time

import sys
sys.path.append("..")

import tools.utils as utils
from tools.small_model import FC_MD
from RicciCurvature.OllivierRicci import OllivierRicci
from tools.FC_linear import FC_Linear
from tools.graph_curvature import graph_curvature_main_torch


np.set_printoptions(threshold=np.inf)
torch.set_printoptions(threshold=torch.inf)

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
seed = 59
    
# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
data_train = MNIST('./data/mnist',
                  train=True,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))

data_test = MNIST('./data/mnist',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))




layers = [2, 4, 5, 6, 7]

model_zoo = {
    2: [784, 20, 15, 10],
    21: [784, 200, 150, 10],
    4: [784, 15, 25, 20, 15, 10],
    5: [784, 20, 30, 30, 20, 15, 10],
    6: [784, 20, 30, 30, 35, 20, 15, 10],
    7: [784, 30, 30, 40, 50, 30, 25, 20, 10]
}

selected_classes = [0,1,2,3,4,5,6,7,8,9]

In [ ]:
def standard_PGD(model, images, labels, device, eps=11/255, alpha=2/255, iters=40):
    images = images.to(device)
    labels = labels.to(device)
    loss = nn.CrossEntropyLoss()
        
    ori_images = images.data
        
    for i in range(iters) :    
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels).to(device)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()
            
    return images


def test(n, loader, eps, alpha, iters, device):    
    n.eval()
    robust_pair = defaultdict(list)
    succ_pair = defaultdict(list)
 
    
    for l in selected_classes:
        for i, (images, labels) in enumerate(loader[l]):
            images = images.to(device)
            labels = labels.to(device)
            output = n(images)
            pred = output.detach().max(1)[1]
            
            adv_img = standard_PGD(n, images, labels, device, eps, alpha, iters)
            adv_out = n(adv_img)
            adv_pred = adv_out.detach().max(1)[1]
 
            robust_l = pred.eq(labels.view_as(pred)) & adv_pred.eq(labels.view_as(pred))
            
            succ_l = pred.eq(labels.view_as(pred)) & ~adv_pred.eq(labels.view_as(pred))
   
            succ_pair[l].append((images[succ_l].cpu(), adv_img[succ_l].cpu()))
            robust_pair[l].append((images[robust_l].cpu(), adv_img[robust_l].cpu()))
            
        print(f'Finish label {l}....')

    return succ_pair, robust_pair

def get_fraction(curvature, b):
    c = []
    neg = 0
    total_e = 0
    for i in range(b):
        ricci_curv = np.array(curvature[i])
        for (i, j, curr) in ricci_curv:
            if curr < 0:
                neg += 1
                total_e += 1
            c.append(curr)
    return neg, total_e, c


In [ ]:
# For FC
def layerwise_shortest_path_torch(dims, weights, device='cpu'):
    batch_size, edge_num = weights.shape
    num_layers = len(dims)
    paths = {}

    weight_idx = 0
    for i in range(num_layers - 1):
        src_size, dst_size = dims[i], dims[i+1]
        direct_dist = weights[:, weight_idx:weight_idx+src_size*dst_size]
        direct_dist = direct_dist.reshape(src_size, dst_size)
        # inf = torch.tensor(float('inf'), device=device)
        # paths[(i, i+1)] = np.where(direct_dist > 0, direct_dist, float('inf'))
        paths[(i, i+1)] = direct_dist
        weight_idx += src_size * dst_size

    return paths

In [ ]:
def vis_nodes(nodes, nodes_val):
    num_nodes = len(nodes[0])
    if num_nodes == 0:
        print("No nodes to visualize.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    # **First Block (Default 28x28 if possible)**
    first_block_size = min(784, num_nodes)
    grid_size = int(np.sqrt(first_block_size))  # Adaptive for non-28x28 cases
    
    sns.heatmap(nodes[0][:first_block_size].reshape(grid_size, grid_size), cmap="coolwarm", ax=axes[0, 0])
    axes[0, 0].set_title(f"Ricci Curvature Nodes (First {grid_size}x{grid_size})")

    sns.heatmap(nodes_val[0][:first_block_size].reshape(grid_size, grid_size), cmap="coolwarm", ax=axes[0, 1])
    axes[0, 1].set_title(f"Original Nodes_Val (First {grid_size}x{grid_size})")

    # **Remaining Nodes (No Reshaping into Square)**
    if num_nodes > first_block_size:
        remaining_nodes = nodes[0][first_block_size:]
        remaining_nodes_val = nodes_val[0][first_block_size:]

        # If remaining nodes are too many, display them as a 1D heatmap.
        if len(remaining_nodes) > 100:
            sns.heatmap(remaining_nodes.reshape(1, -1), cmap="coolwarm", ax=axes[1, 0])
            axes[1, 0].set_title("Ricci Curvature Nodes (Remaining - 1D View)")

            sns.heatmap(remaining_nodes_val.reshape(1, -1), cmap="coolwarm", ax=axes[1, 1])
            axes[1, 1].set_title("Original Nodes_Val (Remaining - 1D View)")
        else:
            # If the remaining nodes fit nicely in 2D, just plot them as a 2D heatmap
            sns.heatmap(remaining_nodes.reshape(-1, 1), cmap="coolwarm", ax=axes[1, 0])
            axes[1, 0].set_title("Ricci Curvature Nodes (Remaining)")

            sns.heatmap(remaining_nodes_val.reshape(-1, 1), cmap="coolwarm", ax=axes[1, 1])
            axes[1, 1].set_title("Original Nodes_Val (Remaining)")

    plt.tight_layout()
    plt.show()



In [ ]:
def vis_edge(curv_p, weight_p):
    # **Heatmap for each layer transition**
    for (layer_start, layer_end) in curv_p.keys():
        # if (layer_start == 0):
        #     continue
        
        fig, axes2 = plt.subplots(1, 2, figsize=(18, 6))

        # Edge Presence Heatmap
        # sns.heatmap(edge_p[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[0])
        # axes2[0].set_title(f"Weight Value: Layer {layer_start} → {layer_end}")

        # Weight Heatmap
        sns.heatmap(curv_p[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[0])
        axes2[0].set_title(f"Ricci Curvature: Layer {layer_start} → {layer_end}")

        # Ricci Curvature Heatmap
        # print(f'Node value for layer {layer_start} - {layer_end} is {weight_p[(layer_start, layer_end)]}')
        sns.heatmap(weight_p[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[1])
        axes2[1].set_title(f"Ricci Curvature ADV: Layer {layer_start} → {layer_end}")

        plt.show()

In [ ]:
def visualization(dims, nodes_val, edges, weight_inv, ricci, b = 1):
    prefix_dims = np.cumsum([0] + dims).tolist()
    nodes = np.zeros(nodes_val.shape)
    
    edge_p = layerwise_shortest_path_torch(dims, edges)
    weight_p = layerwise_shortest_path_torch(dims, weight_inv)
    weight1_p = layerwise_shortest_path_torch(dims, ricci)
    # curv_p = {}
    # curv_adv_p = {}
    
    # for i in range(len(dims) - 1):
    #     src_size, dst_size = dims[i], dims[i+1]
    #     curv_p[(i, i+1)] = np.ones((src_size, dst_size), dtype=np.float32)*2
    #     curv_adv_p[(i, i+1)] = np.ones((src_size, dst_size), dtype=np.float32)*2
        
    # for batch in range(b):
    #     ricci_curv = np.array(ricci[batch])
    #     for (i, j, curr) in ricci_curv:
    #         if curr < 0:
    #             n = int(i)
    #             nodes[0][n] += 1
            
    #         i_layer = np.searchsorted(prefix_dims, i, side='right') - 1
    #         j_layer = np.searchsorted(prefix_dims, j, side='right') - 1
    #         i_idx = int(i - prefix_dims[i_layer])
    #         j_idx = int(j - prefix_dims[j_layer])
            
    #         curv_p[(i_layer, j_layer)][i_idx, j_idx] = curr
            
    # for batch in range(b):
    #     ricci_curv_adv = np.array(weight_inv[batch])
    #     for (i, j, curr) in ricci_curv_adv:
    #         i_layer = np.searchsorted(prefix_dims, i, side='right') - 1
    #         j_layer = np.searchsorted(prefix_dims, j, side='right') - 1
    #         i_idx = int(i - prefix_dims[i_layer])
    #         j_idx = int(j - prefix_dims[j_layer])
            
    #         curv_adv_p[(i_layer, j_layer)][i_idx, j_idx] = curr
            
    # nodes
    # vis_nodes(nodes, nodes_val)
    
    # edges
    vis_edge(weight_p, weight1_p)
    
    

In [ ]:
train_loader, test_loader, valid_loader, valid_dataset, test_dataset = utils.get_new_data(selected_classes, data_train, data_test, test_bs=2000, valid_num=5000)

sep_dataloader = utils.sep_label(test_dataset, selected_classes, bs=2000)

eps = [0.05, 0.07, 0.1, 0.2]
Q = [1]
# eps = [0.1]

model_type = 'fc'
model_pre_name = 'adv'
metric = 'q_inv'
res_path = 'res/' + metric + '/'
model_path = 'pgd/models/'
dataset = 'mnist'
alpha = 0
sample_num = 1

In [ ]:
model_full_n = model_type.lower() + model_pre_name.lower()
sample_size = sample_num

if not os.path.exists(res_path):
    os.makedirs(res_path)
    
layers = [2]
if model_pre_name.lower() == 'big':
    layers = [2]

# build model
for layer_num in layers:
    for q in Q:
        dims = model_zoo[layer_num]
        
        if model_pre_name.lower() == "ori" or model_pre_name.lower() == "decay":
            model_name = "best_ori_10l_" + str(layer_num) + ".pth"
        elif model_pre_name.lower() == "adv":
            model_name = "pgdtrain_" + str(layer_num) + ".pth"
        elif model_pre_name.lower() == 'big':
            model_name = "best_21_adv.pth"
            dims = model_zoo[21]
        else:
            raise Exception("Invalid model name, model name should be {ori, decay, adv}!")
        
                    
        print(f'Now for model {model_name}....\n')

        net_H = FC_MD(dims, layer_num)

        net_H.load_state_dict(torch.load(model_path + model_name))
        net_H = net_H.to(device)
        
        neural_list = []
        nodes_num = 0
        edges_num = 0
        i = 0
        for p in net_H.parameters():
            if i == 0:
                nodes_num += p.shape[1]
            if i%2 == 0:
                nodes_num += p.shape[0]
                edges_num += (p.shape[0] * p.shape[1])
                neural_list.append(p.shape[0])
            i += 1
    
        for e in eps:
            print(f'Current eps {e}: ')
            succ_pair, robust_pair = test(net_H, sep_dataloader, eps=e, alpha=2/255, iters=40, device=device)
            
            robust_c = defaultdict(list)
            nonrobust_c = defaultdict(list)
            non_fraction = defaultdict(list)
            rob_fraction = defaultdict(list)
                
            for l in [2,4,8]:
                print(f'Current label {l}: \n')
                # non robust images
                count = 0
                l_ori = 0.
                l_adv =0.
                
                print(f'Non-Robust pair')
                for (ori_im, adv_im) in succ_pair[l]:
                    for (im, adv) in zip(ori_im, adv_im):
                        img = im.to(device)
                        edge_array, nodes_ori, output, all_node = net_H.NN_info_batch(img.unsqueeze(0))
                       
                        img_adv = adv.to(device)
                        edge_array_adv, nodes_ori_adv, output_adv, all_node_adv = net_H.NN_info_batch(img_adv.unsqueeze(0))
                        
                        # weights = output.detach().clone().to(device)                   
                        # weights[edge_array == 0] = 0.
                        
                        # weights_adv = output_adv.detach().clone().to(device)                   
                        # weights_adv[edge_array_adv == 0] = 0.
                        
                        # weights_inv = net_H.normalization_weight_w2(nodes_ori, weights, dims)
                        # weights_inv = weights_inv.detach()
                        # ricci_curvature = graph_curvature_main_torch(dims, weights_inv, device=device, alpha=alpha)
                        
                        # weights_inv_adv = net_H.normalization_weight_w2(nodes_ori_adv, weights_adv, dims)
                        # weights_inv_adv = weights_inv_adv.detach()
                        # ricci_curvature_adv = graph_curvature_main_torch(dims, weights_inv_adv, device=device, alpha=alpha)
            
                        # print(f'Label {l}: non-robust img')
                        w = output.detach().cpu().numpy()  
                        w1 = output_adv.detach().cpu().numpy()
                        n = all_node.detach().cpu().numpy()
                        n1 = all_node_adv.detach().cpu().numpy()
                        w[abs(n) < 0.1] = 0.
                        w1[abs(n1) < 0.1] = 0.
                        w[abs(w) > 0.3] = 0.
                        w1[abs(w1) > 0.3] = 0.
                        # ww = w[abs(n) < 0.1]
                        # ww1 = w1[abs(n1) < 0.1]
                        # l_ori += len(ww[abs(ww) > 0.3])
                        # l_adv += len(ww1[abs(ww1) > 0.3])
                        # print(f'Ori example weights: {len(ww[abs(ww) > 0.70])}, adv example weights: {len(ww1[abs(ww1) > 0.70])}')
                        # print(f'The ori example has {len(ww)} zero nodes, adv example has {len(ww1)} zero nodes')
                        # print(f'Ori example node value: {np.min(n[n!=0])} - {np.max(n)}, average is {np.mean(n)}; adv example node value: {np.min(n1[n1!=0])} - {np.max(n1)}, average is {np.mean(n1)}')
                        # print(f'Ori example weights: {np.min(ww[ww!=0])} - {np.max(ww)}, average is {np.mean(ww)}; adv example weights: {np.min(ww1[ww1!=0])} - {np.max(ww1)}, average is {np.mean(ww1)}\n')
                        visualization(dims, nodes_ori.detach().cpu().numpy(), output.detach().cpu().numpy(), w, w1)
                        # print(f'Label {l}: non-robust Adv img')
                        # visualization(dims, nodes_ori_adv.detach().cpu().numpy(), output_adv.detach().cpu().numpy(), all_node_adv.detach().cpu().numpy(), ricci_curvature_adv, weights_inv_adv.shape[0])
                        
                        count += 1
                        if (count % 10 == 0):
                            print(f'Finish {count} graphs....')
                            
                        if (count >= sample_size):
                            break
                # l_ori = l_ori/count if count > 0 else l_ori
                # l_adv = l_adv/count if count > 0 else l_adv
                # print(f'Label {l} - eps {e}: non-robust img: the average number of ori example is {l_ori}, adv example is {l_adv}\n')


                print(f'Robust pair')
                # robust images
                count = 0
                l_ori = 0.
                l_adv =0.
                for (ori_im, adv_im) in robust_pair[l]:
                    for (im, adv) in zip(ori_im, adv_im):
                        img = im.to(device)
                        edge_array, nodes_ori, output, all_node = net_H.NN_info_batch(img.unsqueeze(0))
                        img_adv = adv.to(device)
                        edge_array_adv, nodes_ori_adv, output_adv, all_node_adv = net_H.NN_info_batch(img_adv.unsqueeze(0))
                        
                        # weights = output.detach().clone().to(device)                   
                        # weights[edge_array == 0] = 0.
                        
                        # weights_adv = output_adv.detach().clone().to(device)                   
                        # weights_adv[edge_array_adv == 0] = 0.
                        
                        # weights_inv = net_H.normalization_weight_w2(nodes_ori, weights, dims)
                        # weights_inv = weights_inv.detach()
                        # ricci_curvature = graph_curvature_main_torch(dims, weights_inv, device=device, alpha=alpha)
                        
                        # weights_inv_adv = net_H.normalization_weight_w2(nodes_ori_adv, weights_adv, dims)
                        # weights_inv_adv = weights_inv_adv.detach()
                        # ricci_curvature_adv = graph_curvature_main_torch(dims, weights_inv_adv, device=device, alpha=alpha)
            
                        # print(f'Label {l}: robust img')
                        w = output.detach().cpu().numpy()  
                        w1 = output_adv.detach().cpu().numpy()
                        n = all_node.detach().cpu().numpy()
                        n1 = all_node_adv.detach().cpu().numpy()
                        # ww = w[abs(n) < 0.1]
                        # ww1 = w1[abs(n1) < 0.1]
                        # l_ori += len(ww[abs(ww) > 0.3])
                        # l_adv += len(ww1[abs(ww1) > 0.3])
                        w[abs(n) < 0.1] = 0.
                        w1[abs(n1) < 0.1] = 0.
                        w[abs(w) > 0.3] = 0.
                        w1[abs(w1) > 0.3] = 0.
                        # print(f'Ori example weights: {len(ww[abs(ww) > 0.7])}, adv example weights: {len(ww1[abs(ww1) > 0.7])}')
                        # print(f'The ori example has {len(ww)} zero nodes, adv example has {len(ww1)} zero nodes')
                        # print(f'Ori example node value: {np.min(n[n!=0])} - {np.max(n)}, average is {np.mean(n)}; adv example node value: {np.min(n1[n1!=0])} - {np.max(n1)}, average is {np.mean(n1)}')
                        # print(f'Ori example weights: {np.min(ww[ww!=0])} - {np.max(ww)}, average is {np.mean(ww)}; adv example weights: {np.min(ww1[ww1!=0])} - {np.max(ww1)}, average is {np.mean(ww1)}\n')
                        # print(f'Ori example weights: {np.min(w[w!=0])} - {np.max(w)}, average is {np.mean(w)}; adv example weights: {np.min(w1[w1!=0])} - {np.max(w1)}, average is {np.mean(w1)}\n')
                        visualization(dims, nodes_ori.detach().cpu().numpy(), output.detach().cpu().numpy(), w, w1)
                        # print(f'Label {l}: robust Adv example:')
                        # visualization(dims, nodes_ori_adv.detach().cpu().numpy(), output_adv.detach().cpu().numpy(), all_node_adv.detach().cpu().numpy(), ricci_curvature_adv, weights_inv_adv.shape[0])
                        
                        count += 1
                        if (count % 10 == 0):
                            print(f'Finish {count} graphs....')
                            
                        if (count >= sample_size):
                            break
                        
                # l_ori = l_ori/count if count > 0 else l_ori
                # l_adv = l_adv/count if count > 0 else l_adv
                # print(f'Label {l} - eps {e}: robust img: the average number of ori example is {l_ori}, adv example is {l_adv}\n')